<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Finance-KnowledgeOps-Copilot/blob/main/Enterprise_Finance_KnowledgeOps_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy pyspark sentence-transformers faiss-cpu langchain langchain-community transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip -q install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [4]:
import os
import json
import random
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [5]:
random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Enterprise-Finance-KnowledgeOps-Copilot")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
DAG_DIR = PROJECT_DIR / "dags"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, PROCESSED_DIR, VECTOR_DIR, OUTPUT_DIR, DAG_DIR, NOTEBOOK_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project root:", PROJECT_DIR.resolve())

Project folders created successfully.
Project root: /content/Enterprise-Finance-KnowledgeOps-Copilot


In [7]:
vendors = pd.DataFrame([
    ["V001", "Alpha Energy Services", "Maintenance", "NY", "Active"],
    ["V002", "BlueGrid Technologies", "IT Services", "CT", "Active"],
    ["V003", "Northwind Electric", "Electrical Components", "MA", "Active"],
    ["V004", "GreenVolt Contractors", "Field Services", "PA", "Active"],
    ["V005", "Prime Utility Logistics", "Logistics", "NJ", "Review"],
    ["V006", "CoreAxis Industrial Supply", "Industrial Supply", "MI", "Active"]
], columns=["vendor_id", "vendor_name", "vendor_category", "vendor_state", "vendor_status"])


gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["500500", "Opex - Field Operations", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])


cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Ops"],
    ["CC140", "Finance Operations", "Finance"],
    ["CC150", "Grid Modernization", "Engineering"]
], columns=["cost_center_id", "cost_center_name", "department"])


finance_policies = pd.DataFrame([
    ["POL001", "Invoice Policy", "Invoices above 40000 USD require finance manager approval before payment release.", "High"],
    ["POL002", "Vendor Compliance Policy", "Invoices from vendors in review status must be flagged for manual compliance verification.", "High"],
    ["POL003", "Duplicate Control Policy", "Invoices with same vendor, amount, and close invoice date must be checked for duplicate risk.", "Medium"],
    ["POL004", "Missing Data Policy", "Invoices with missing vendor, missing description, or invalid amount must be blocked from downstream approval workflows.", "High"],
    ["POL005", "Late Payment Policy", "Payments delayed beyond 30 days from posting date should be investigated for operational or financial control gaps.", "Medium"]
], columns=["policy_id", "policy_name", "policy_text", "severity"])


vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)
finance_policies.to_csv(RAW_DIR / "finance_policies.csv", index=False)

print("Master data created successfully.")
print("Files saved:")
print("-", RAW_DIR / "vendors.csv")
print("-", RAW_DIR / "gl_accounts.csv")
print("-", RAW_DIR / "cost_centers.csv")
print("-", RAW_DIR / "finance_policies.csv")

Master data created successfully.
Files saved:
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/vendors.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/gl_accounts.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/cost_centers.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/finance_policies.csv


In [8]:
invoice_rows = []
payment_rows = []

start_date = datetime(2024, 1, 1)

descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection",
    "grid modernization consulting",
    "vendor maintenance support"
]

for i in range(1, 151):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 10))
    amount = round(random.uniform(1000, 60000), 2)

    source_doc_id = f"SRC{i:04d}"
    invoice_id = f"INV{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        amount,
        random.choice(descriptions)
    ])

    payment_delay = random.randint(5, 45)
    payment_date = posting_date + timedelta(days=payment_delay)

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        round(amount * random.uniform(0.95, 1.00), 2)
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "vendor_id", "gl_account", "cost_center_id",
    "invoice_date", "posting_date", "amount_usd", "description"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id", "payment_date", "paid_amount_usd"
])

# Add enterprise-style data quality and control issues
invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500
invoices.loc[10, "description"] = None

# duplicate-like scenario
invoices.loc[20, ["vendor_id", "amount_usd", "invoice_date", "description"]] = [
    invoices.loc[21, "vendor_id"],
    invoices.loc[21, "amount_usd"],
    invoices.loc[21, "invoice_date"],
    invoices.loc[21, "description"]
]

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Transaction data created successfully.")
print("Invoices shape:", invoices.shape)
print("Payments shape:", payments.shape)
display(invoices.head())
display(payments.head())

Transaction data created successfully.
Invoices shape: (150, 9)
Payments shape: (150, 5)


,invoice_id,source_doc_id,vendor_id,gl_account,cost_center_id,invoice_date,posting_date,amount_usd,description
0,INV00001,SRC0001,V001,500300,CC140,2024-11-21,2024-11-23,2475.63,it platform support
1,INV00002,SRC0002,V001,500200,CC150,2024-02-13,2024-02-22,6129.39,distribution network inspection
2,INV00003,SRC0003,V001,400100,CC100,2024-04-17,2024-04-21,30815.96,transformer repair service
3,INV00004,SRC0004,None,500500,CC150,2024-11-19,2024-11-28,25751.67,grid modernization consulting
4,INV00005,SRC0005,V004,500400,CC140,2024-01-01,2024-01-04,42190.22,logistics support for equipment


,payment_id,source_doc_id,invoice_id,payment_date,paid_amount_usd
0,PAY00001,SRC0001,INV00001,2024-12-13,2379.48
1,PAY00002,SRC0002,INV00002,2024-02-29,5832.05
2,PAY00003,SRC0003,INV00003,2024-05-31,29581.53
3,PAY00004,SRC0004,INV00004,2025-01-09,24822.28
4,PAY00005,SRC0005,INV00005,2024-01-26,40408.69


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, when, trim, upper, datediff, to_date
)

spark = SparkSession.builder.appName("EnterpriseFinanceKnowledgeOpsCopilot").getOrCreate()

base_path = str(RAW_DIR)
processed_path = str(PROCESSED_DIR)
output_path = str(OUTPUT_DIR)

vendors_spark = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts_spark = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers_spark = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
finance_policies_spark = spark.read.option("header", True).csv(f"{base_path}/finance_policies.csv")
invoices_spark = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments_spark = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

# Standardization
vendors_spark = vendors_spark.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices_spark = invoices_spark.withColumn("description_clean", upper(trim(col("description"))))

# Common business key
invoices_spark = invoices_spark.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

# Integrated finance view
integrated = (
    invoices_spark
    .join(vendors_spark, on="vendor_id", how="left")
    .join(gl_accounts_spark, on="gl_account", how="left")
    .join(cost_centers_spark, on="cost_center_id", how="left")
    .join(
        payments_spark.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

# Date conversion for control checks
integrated = (
    integrated
    .withColumn("posting_date_dt", to_date(col("posting_date")))
    .withColumn("payment_date_dt", to_date(col("payment_date")))
    .withColumn("invoice_date_dt", to_date(col("invoice_date")))
    .withColumn("payment_delay_days", datediff(col("payment_date_dt"), col("posting_date_dt")))
)

# Finance risk / control flags
integrated = (
    integrated
    .withColumn(
        "high_value_flag",
        when(col("amount_usd").cast("double") > 40000, lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "vendor_review_flag",
        when(col("vendor_status") == "Review", lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "late_payment_flag",
        when(col("payment_delay_days") > 30, lit("Y")).otherwise(lit("N"))
    )
)

# Data quality checks
dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

# Retrieval-ready finance narrative
retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Vendor Status"),
        col("vendor_status"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description"),
        lit("High Value Flag"),
        col("high_value_flag"),
        lit("Late Payment Flag"),
        col("late_payment_flag")
    )
)

embedding_input = retrieval_docs.select(
    "invoice_id",
    "common_business_key",
    "vendor_id",
    "vendor_name",
    "cost_center_id",
    "cost_center_name",
    "gl_account",
    "gl_account_name",
    "amount_usd",
    "retrieval_text"
)

print("Integrated finance view:")
integrated.show(5, truncate=False)

print("Data quality issues:")
dq.show(truncate=False)

print("Embedding input:")
embedding_input.show(5, truncate=False)

Integrated finance view:
+----------+--------------+----------+---------+-------------+------------+------------+----------+-------------------------------+-------------------------------+-------------------+---------------------+---------------+------------+-------------+---------------------+-------------------------+-------+-----------------------+-----------+----------+------------+---------------+---------------+---------------+---------------+------------------+---------------+------------------+-----------------+
|invoice_id|cost_center_id|gl_account|vendor_id|source_doc_id|invoice_date|posting_date|amount_usd|description                    |description_clean              |common_business_key|vendor_name          |vendor_category|vendor_state|vendor_status|vendor_name_clean    |gl_account_name          |gl_type|cost_center_name       |department |payment_id|payment_date|paid_amount_usd|posting_date_dt|payment_date_dt|invoice_date_dt|payment_delay_days|high_value_flag|vendor_revi

In [10]:
integrated_pd = integrated.toPandas()
dq_pd = dq.toPandas()
embedding_input_pd = embedding_input.toPandas()
finance_policies_pd = finance_policies_spark.toPandas()

integrated_pd.to_csv(PROCESSED_DIR / "integrated_finance_view.csv", index=False)
dq_pd.to_csv(OUTPUT_DIR / "data_quality_issues.csv", index=False)
embedding_input_pd.to_csv(PROCESSED_DIR / "embedding_input.csv", index=False)
finance_policies_pd.to_csv(PROCESSED_DIR / "finance_policies.csv", index=False)

print("Processed files saved successfully.\n")
print("Saved files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    print("-", p.name)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)

Processed files saved successfully.

Saved files:
- embedding_input.csv
- finance_policies.csv
- integrated_finance_view.csv
- data_quality_issues.csv


In [11]:
from sentence_transformers import SentenceTransformer

embedding_input_df = pd.read_csv(PROCESSED_DIR / "embedding_input.csv")
finance_policies_df = pd.read_csv(PROCESSED_DIR / "finance_policies.csv")

finance_record_docs = []
for _, row in embedding_input_df.iterrows():
    finance_record_docs.append({
        "doc_id": f"FIN_{row['invoice_id']}",
        "doc_type": "finance_record",
        "source_id": row["invoice_id"],
        "common_business_key": row["common_business_key"],
        "text": row["retrieval_text"],
        "metadata": {
            "invoice_id": row["invoice_id"],
            "vendor_id": row["vendor_id"],
            "vendor_name": row["vendor_name"],
            "cost_center_id": row["cost_center_id"],
            "cost_center_name": row["cost_center_name"],
            "gl_account": row["gl_account"],
            "gl_account_name": row["gl_account_name"],
            "amount_usd": row["amount_usd"]
        }
    })

policy_docs = []
for _, row in finance_policies_df.iterrows():
    policy_docs.append({
        "doc_id": f"POL_{row['policy_id']}",
        "doc_type": "finance_policy",
        "source_id": row["policy_id"],
        "common_business_key": "POLICY",
        "text": f"Policy Name: {row['policy_name']}. Policy Text: {row['policy_text']}. Severity: {row['severity']}.",
        "metadata": {
            "policy_id": row["policy_id"],
            "policy_name": row["policy_name"],
            "severity": row["severity"]
        }
    })

all_docs = finance_record_docs + policy_docs

documents_df = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "doc_type": d["doc_type"],
        "source_id": d["source_id"],
        "common_business_key": d["common_business_key"],
        "text": d["text"],
        "metadata_json": json.dumps(d["metadata"])
    }
    for d in all_docs
])

documents_df.to_csv(PROCESSED_DIR / "rag_documents.csv", index=False)

print("RAG documents prepared successfully.")
print("Total documents:", len(documents_df))
display(documents_df.head())

RAG documents prepared successfully.
Total documents: 155


,doc_id,doc_type,source_id,common_business_key,text,metadata_json
0,FIN_INV00001,finance_record,INV00001,V001_CC140_500300,Invoice INV00001 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00001"", ""vendor_id"": ""V001""..."
1,FIN_INV00002,finance_record,INV00002,V001_CC150_500200,Invoice INV00002 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00002"", ""vendor_id"": ""V001""..."
2,FIN_INV00003,finance_record,INV00003,V001_CC100_400100,Invoice INV00003 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00003"", ""vendor_id"": ""V001""..."
3,FIN_INV00004,finance_record,INV00004,CC150_500500,Invoice INV00004 Vendor Vendor Status Cost Cen...,"{""invoice_id"": ""INV00004"", ""vendor_id"": NaN, ""..."
4,FIN_INV00005,finance_record,INV00005,V004_CC140_500400,Invoice INV00005 Vendor GreenVolt Contractors ...,"{""invoice_id"": ""INV00005"", ""vendor_id"": ""V004""..."


In [12]:
documents_df = pd.read_csv(PROCESSED_DIR / "rag_documents.csv")

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

document_texts = documents_df["text"].fillna("").tolist()
document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(VECTOR_DIR / "document_embeddings.npy", document_embeddings)
documents_df.to_csv(VECTOR_DIR / "vector_documents.csv", index=False)

print("Embeddings generated successfully.")
print("Embedding model:", embedding_model_name)
print("Embeddings shape:", document_embeddings.shape)
display(documents_df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings generated successfully.
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embeddings shape: (155, 384)


,doc_id,doc_type,source_id,common_business_key,text,metadata_json
0,FIN_INV00001,finance_record,INV00001,V001_CC140_500300,Invoice INV00001 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00001"", ""vendor_id"": ""V001""..."
1,FIN_INV00002,finance_record,INV00002,V001_CC150_500200,Invoice INV00002 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00002"", ""vendor_id"": ""V001""..."
2,FIN_INV00003,finance_record,INV00003,V001_CC100_400100,Invoice INV00003 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00003"", ""vendor_id"": ""V001""..."
3,FIN_INV00004,finance_record,INV00004,CC150_500500,Invoice INV00004 Vendor Vendor Status Cost Cen...,"{""invoice_id"": ""INV00004"", ""vendor_id"": NaN, ""..."
4,FIN_INV00005,finance_record,INV00005,V004_CC140_500400,Invoice INV00005 Vendor GreenVolt Contractors ...,"{""invoice_id"": ""INV00005"", ""vendor_id"": ""V004""..."


In [13]:
import faiss

documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")
document_embeddings = np.load(VECTOR_DIR / "document_embeddings.npy")

embedding_matrix = document_embeddings.astype("float32")

faiss_index = faiss.IndexFlatL2(embedding_matrix.shape[1])
faiss_index.add(embedding_matrix)

faiss.write_index(faiss_index, str(VECTOR_DIR / "finance_knowledge.index"))

print("FAISS index created successfully.")
print("Number of documents indexed:", faiss_index.ntotal)
print("Embedding dimension:", embedding_matrix.shape[1])

FAISS index created successfully.
Number of documents indexed: 155
Embedding dimension: 384


In [14]:
documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")
faiss_index = faiss.read_index(str(VECTOR_DIR / "finance_knowledge.index"))

def search_finance_docs(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        row = documents_df.iloc[idx]
        results.append({
            "rank": rank + 1,
            "doc_id": row["doc_id"],
            "doc_type": row["doc_type"],
            "source_id": row["source_id"],
            "common_business_key": row["common_business_key"],
            "text": row["text"],
            "distance": float(distances[0][rank])
        })
    return pd.DataFrame(results)

query = "Show me high value invoices that may need finance approval and policy guidance"
search_results = search_finance_docs(query, top_k=5)

print("Query:", query)
display(search_results)

Query: Show me high value invoices that may need finance approval and policy guidance


,rank,doc_id,doc_type,source_id,common_business_key,text,distance
0,1,POL_POL001,finance_policy,POL001,POLICY,Policy Name: Invoice Policy. Policy Text: Invo...,0.627424
1,2,FIN_INV00148,finance_record,INV00148,V005_CC140_500500,Invoice INV00148 Vendor Prime Utility Logistic...,0.903783
2,3,FIN_INV00117,finance_record,INV00117,V005_CC150_500500,Invoice INV00117 Vendor Prime Utility Logistic...,0.912950
3,4,FIN_INV00057,finance_record,INV00057,V005_CC110_500500,Invoice INV00057 Vendor Prime Utility Logistic...,0.913464
4,5,FIN_INV00088,finance_record,INV00088,V004_CC140_500100,Invoice INV00088 Vendor GreenVolt Contractors ...,0.940382


In [17]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")

langchain_docs = []
for _, row in documents_df.iterrows():
    metadata = json.loads(row["metadata_json"]) if pd.notna(row["metadata_json"]) else {}
    metadata["doc_id"] = row["doc_id"]
    metadata["doc_type"] = row["doc_type"]
    metadata["source_id"] = row["source_id"]
    metadata["common_business_key"] = row["common_business_key"]

    langchain_docs.append(
        Document(
            page_content=row["text"],
            metadata=metadata
        )
    )

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

langchain_vectorstore = FAISS.from_documents(
    documents=langchain_docs,
    embedding=hf_embeddings
)

langchain_vectorstore.save_local(str(VECTOR_DIR / "langchain_faiss_store"))

print("LangChain FAISS vector store created successfully.")
print("Total LangChain documents:", len(langchain_docs))
print("Saved path:", VECTOR_DIR / "langchain_faiss_store")
print("\nSample metadata:")
print(langchain_docs[0].metadata)
print("\nSample text:")
print(langchain_docs[0].page_content[:300])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LangChain FAISS vector store created successfully.
Total LangChain documents: 155
Saved path: Enterprise-Finance-KnowledgeOps-Copilot/data/vector_store/langchain_faiss_store

Sample metadata:
{'invoice_id': 'INV00001', 'vendor_id': 'V001', 'vendor_name': 'Alpha Energy Services', 'cost_center_id': 'CC140', 'cost_center_name': 'Finance Operations', 'gl_account': 500300, 'gl_account_name': 'Opex - Contractors', 'amount_usd': 2475.63, 'doc_id': 'FIN_INV00001', 'doc_type': 'finance_record', 'source_id': 'INV00001', 'common_business_key': 'V001_CC140_500300'}

Sample text:
Invoice INV00001 Vendor Alpha Energy Services Vendor Status Active Cost Center Finance Operations GL Account Opex - Contractors Amount 2475.63 Description it platform support High Value Flag N Late Payment Flag N


In [19]:
retriever = langchain_vectorstore.as_retriever(search_kwargs={"k": 5})

query = "Find high value invoices, vendor review cases, and related finance approval policy"

retrieved_docs = retriever.invoke(query)

print("Query:", query)
print("\nTop retrieved LangChain documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content[:500])
    print("-" * 100)

Query: Find high value invoices, vendor review cases, and related finance approval policy

Top retrieved LangChain documents:

Result 1
Metadata: {'policy_id': 'POL001', 'policy_name': 'Invoice Policy', 'severity': 'High', 'doc_id': 'POL_POL001', 'doc_type': 'finance_policy', 'source_id': 'POL001', 'common_business_key': 'POLICY'}
Text: Policy Name: Invoice Policy. Policy Text: Invoices above 40000 USD require finance manager approval before payment release.. Severity: High.
----------------------------------------------------------------------------------------------------
Result 2
Metadata: {'policy_id': 'POL002', 'policy_name': 'Vendor Compliance Policy', 'severity': 'High', 'doc_id': 'POL_POL002', 'doc_type': 'finance_policy', 'source_id': 'POL002', 'common_business_key': 'POLICY'}
Text: Policy Name: Vendor Compliance Policy. Policy Text: Invoices from vendors in review status must be flagged for manual compliance verification.. Severity: High.
-------------------------------------

In [20]:
from transformers import pipeline

# Load lightweight Hugging Face model (safe for Colab)
hf_llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=512,
    temperature=0.2
)

def generate_finance_response(query):
    # Step 1: Retrieve relevant docs
    retrieved_docs = retriever.invoke(query)

    # Step 2: Combine context
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Step 3: Prompt
    prompt = f"""
You are an enterprise finance assistant.

Use the below context to answer the question.
Give a clear business explanation and mention any risks or policies.

Context:
{context}

Question:
{query}

Answer:
"""

    # Step 4: Generate response
    response = hf_llm(prompt)[0]["generated_text"]

    return response

# Test query
query = "Why is this invoice risky and does it need approval?"
response = generate_finance_response(query)

print("Query:", query)
print("\nAI Response:\n")
print(response)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 

Query: Why is this invoice risky and does it need approval?

AI Response:


You are an enterprise finance assistant.

Use the below context to answer the question.
Give a clear business explanation and mention any risks or policies.

Context:
Policy Name: Invoice Policy. Policy Text: Invoices above 40000 USD require finance manager approval before payment release.. Severity: High.

Invoice INV00009 Vendor BlueGrid Technologies Vendor Status Active Cost Center Grid Modernization GL Account Opex - Logistics Amount 40014.54 Description it platform support High Value Flag Y Late Payment Flag N

Invoice INV00073 Vendor BlueGrid Technologies Vendor Status Active Cost Center Transmission Operations GL Account Revenue - Energy Delivery Amount 20244.79 Description it platform support High Value Flag N Late Payment Flag Y

Invoice INV00117 Vendor Prime Utility Logistics Vendor Status Review Cost Center Grid Modernization GL Account Opex - Field Operations Amount 19066.16 Description logistics su